# Phase 1 — Denoising Autoencoder Pretraining
**Continuous Sign Language Translation** · Phase 1 of 2

Trains a `MultiStreamSemanticEncoder` + `StructuredDiffusionDecoder` using a proper DDPM noise schedule with epsilon prediction.

The encoder learns a compact latent `Z [B, T/4, 512]` that will be used in Phase 2 for translation.

**Runtime:** GPU (T4 or better)


In [ ]:
# ── Clone repository and install dependencies ──────────────────────────
!git clone https://github.com/bencejdanko/continuous-sign-language-translation.git cslt
%cd cslt
!pip install -q -r requirements.txt

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Hugging Face authentication ───────────────────────────────────────
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your HF token: ")

import os
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

## ⚙️ Configuration
Adjust these before running. All parameters are passed to the `phase1_pretrain` function via `Phase1Config`.


In [ ]:
from config import Phase1Config

cfg = Phase1Config(
    max_samples=100,          # Set to None for full dataset
    batch_size=32,
    epochs=3,
    lr=1e-4,
    ckpt_dir="/content/phase1_ckpt",
    upload_hf=False,           # Set to True to auto-upload checkpoints
    hf_repo="bdanko/continuous-sign-language-translation",
    seed=42,
)
print(f"Config: {cfg}")

## Training
Runs the Phase 1 pretraining loop with DDPM epsilon prediction, configurable masking, train/val splits, and best checkpoint saving.

In [ ]:
from phase1_pretrain import train_phase1

train_phase1(cfg)

## Upload to Hugging Face (optional)

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder

if cfg.upload_hf:
    create_repo(cfg.hf_repo, token=HF_TOKEN, exist_ok=True)
    upload_folder(
        folder_path=cfg.ckpt_dir,
        repo_id=cfg.hf_repo,
        token=HF_TOKEN,
        commit_message=f"Phase 1 checkpoint — {cfg.epochs} epochs, {cfg.max_samples} clips",
    )
    print(f"Uploaded to https://huggingface.co/{cfg.hf_repo} ✓")
else:
    print("Skipping upload. Set cfg.upload_hf = True to upload.")